# FASE 5: Explainable AI (Interpretasi SHAP)
## Prediksi PM2.5 di Cekungan Bandung
**Kerja Praktik — PRSDI BRIN 2026**

---

Notebook ini adalah **puncak pembuktian riset**. Model *Deep Learning* terbaik (LSTM atau BiLSTM) dari Notebook 04 terkenal sebagai *Black Box*: mampu memprediksi dengan akurat, namun tidak transparan.

Dengan metode **SHAP (*SHapley Additive exPlanations*)**, kita membongkar isi model dan mengungkap **faktor meteorologi apa yang paling memicu lonjakan PM2.5** di Cekungan Bandung.


## 1. Import Library


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import joblib
import shap
import tensorflow as tf
from tensorflow.keras.models import load_model

import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.figsize': (14, 6), 'font.size': 11})
print(f"[OK] Library loaded. SHAP versi: {shap.__version__}")


## 2. Load Data, Model Terbaik & Transformasi 3D


In [ ]:
DATA_DIR  = '/content/drive/MyDrive/KP_BRIN/data/processed'
MODEL_DIR = '/content/drive/MyDrive/KP_BRIN/models'

# Load model DL terbaik dari Notebook 04
model = load_model(os.path.join(MODEL_DIR, 'best_dl_model.keras'))
best_info = joblib.load(os.path.join(MODEL_DIR, 'best_dl_info.pkl'))
print(f"[OK] Model dimuat: {best_info['model_name']} (RMSE={best_info['rmse']:.4f})")

# Load Data
train_scaled = pd.read_csv(os.path.join(DATA_DIR, 'train_scaled.csv'), index_col=0, parse_dates=True)
test_scaled  = pd.read_csv(os.path.join(DATA_DIR, 'test_scaled.csv'), index_col=0, parse_dates=True)

target_col = 'pm25'
feature_names = [c for c in train_scaled.columns if c != target_col]

# Transformasi 3D (Sliding Window)
def create_sequences(df, target_col, time_steps=24):
    X, y = [], []
    data_array = df.values
    target_idx = df.columns.get_loc(target_col)
    for i in range(len(data_array) - time_steps):
        X.append(data_array[i:(i + time_steps)])
        y.append(data_array[i + time_steps, target_idx])
    return np.array(X), np.array(y)

TIME_STEPS = 24
X_train_seq, _ = create_sequences(train_scaled, target_col, TIME_STEPS)
X_test_seq, _  = create_sequences(test_scaled, target_col, TIME_STEPS)

print(f"Tensor Input SHAP: {X_test_seq.shape}")
print(f"Jumlah fitur: {len(feature_names)}")
print(f"Fitur: {feature_names}")


## 3. Komputasi SHAP (KernelExplainer + Wrapper 2D)

Karena `DeepExplainer` dan `GradientExplainer` mengalami *bug* di TensorFlow 2.15+ (Keras 3), kita menggunakan **KernelExplainer** yang bersifat *model-agnostic* dan stabil untuk sembarang arsitektur.

KernelExplainer hanya menerima data 2D, sehingga diperlukan fungsi *wrapper* yang secara transparan mengubah data 2D menjadi Tensor 3D saat memanggil model.


In [ ]:
print("Memulai komputasi SHAP (KernelExplainer)...")
print("Estimasi waktu: 3-10 menit tergantung spesifikasi.\n")

# 1. Background Sample (referensi titik nol bagi SHAP)
np.random.seed(42)
background_idx = np.random.choice(X_train_seq.shape[0], 50, replace=False)
background = X_train_seq[background_idx]

# 2. Test Sample (data yang akan dianalisis)
test_idx = np.random.choice(X_test_seq.shape[0], 50, replace=False)
test_sample = X_test_seq[test_idx]

# 3. Fungsi Wrapper 3D -> 2D
def lstm_predict_wrapper(x_2d_flat):
    x_3d = x_2d_flat.reshape(-1, TIME_STEPS, X_train_seq.shape[2])
    return model.predict(x_3d, verbose=0).flatten()

# 4. Flatten data untuk KernelExplainer
background_2d   = background.reshape(background.shape[0], -1)
test_sample_2d  = test_sample.reshape(test_sample.shape[0], -1)

# 5. Inisialisasi dan hitung SHAP values
explainer   = shap.KernelExplainer(lstm_predict_wrapper, background_2d)
shap_values = explainer.shap_values(test_sample_2d)

# Kembalikan ke 3D: [50 sampel, 24 timesteps, n fitur]
shap_values_3d = shap_values.reshape(-1, TIME_STEPS, X_train_seq.shape[2])

print(f"\n[SELESAI] Bentuk matriks SHAP 3D: {shap_values_3d.shape}")


## 4. Flattening Waktu (Meratakan 24 Jam)

Hasil SHAP berbentuk 3D `(50, 24, n_fitur)`. Untuk visualisasi yang mudah dibaca, kita **merata-ratakan** efek sepanjang 24 jam tersebut menjadi 2D `(50, n_fitur)`, sehingga menjawab pertanyaan:

> *"Secara rata-rata dalam 24 jam terakhir, fitur mana yang paling berpengaruh?"*


In [ ]:
# Rata-ratakan pada sumbu waktu (axis=1)
shap_values_2d  = np.mean(shap_values_3d, axis=1)
test_sample_2d_feat = np.mean(test_sample, axis=1)

print(f"Bentuk SHAP (siap plot): {shap_values_2d.shape}")
print(f"Bentuk fitur (siap plot): {test_sample_2d_feat.shape}")


## 5. Visualisasi 1: SHAP Summary Plot (Global Importance)

Grafik Beeswarm ini adalah visualisasi terpenting untuk Bab Hasil:
- **Sumbu Y:** Daftar fitur dari yang terpenting (atas) ke paling tidak penting (bawah)
- **Sumbu X:** Nilai SHAP (kanan = mendorong PM2.5 naik, kiri = mendorong turun)
- **Warna:** Merah = nilai fitur tinggi, Biru = rendah


In [ ]:
plt.figure(figsize=(10, 8))
plt.title(f"SHAP Summary Plot ({best_info['model_name']})\n"
          f"Faktor Pendorong PM2.5 di Cekungan Bandung\n",
          fontsize=14, fontweight='bold')

shap.summary_plot(
    shap_values_2d,
    test_sample_2d_feat,
    feature_names=feature_names,
    plot_type="dot",
    show=False
)
plt.tight_layout()
plt.show()


## 6. Visualisasi 2: SHAP Bar Plot (Mean Absolute Importance)

Grafik batang yang menampilkan rata-rata nilai absolut SHAP per fitur — menunjukkan **besarnya pengaruh** tanpa memperhatikan arah.


In [ ]:
plt.figure(figsize=(10, 8))
plt.title(f"SHAP Feature Importance ({best_info['model_name']})\n",
          fontsize=14, fontweight='bold')

shap.summary_plot(
    shap_values_2d,
    test_sample_2d_feat,
    feature_names=feature_names,
    plot_type="bar",
    show=False
)
plt.tight_layout()
plt.show()


## 7. Visualisasi 3: Dependence Plot

Menyoroti hubungan non-linear antara satu variabel tertentu (misalnya Suhu) dan kontribusinya terhadap kenaikan/penurunan PM2.5.


In [ ]:
# Dependence plot untuk beberapa fitur kunci
key_features = ['suhu_2m_C', 'kelembapan_persen', 'kecepatan_angin_kmh']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'SHAP Dependence Plots ({best_info["model_name"]})',
             fontsize=14, fontweight='bold')

for ax_idx, feat in enumerate(key_features):
    if feat in feature_names:
        feat_idx = feature_names.index(feat)
        plt.sca(axes[ax_idx])
        shap.dependence_plot(
            feat_idx,
            shap_values_2d,
            test_sample_2d_feat,
            feature_names=feature_names,
            interaction_index=None,
            show=False,
            ax=axes[ax_idx]
        )
        axes[ax_idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 8. Simpan Hasil SHAP

Menyimpan matriks SHAP values untuk keperluan analisis lanjutan atau penulisan laporan.


In [ ]:
OUTPUT_DIR = '/content/drive/MyDrive/KP_BRIN/data/processed'

# Simpan SHAP values 2D (sudah di-flatten)
shap_df = pd.DataFrame(shap_values_2d, columns=feature_names)
shap_df.to_csv(os.path.join(OUTPUT_DIR, 'shap_values_2d.csv'), index=False)

# Simpan SHAP values 3D (mentah)
np.save(os.path.join(OUTPUT_DIR, 'shap_values_3d.npy'), shap_values_3d)

print(f"[OK] Hasil SHAP berhasil disimpan.")
print(f"  -> shap_values_2d.csv  (Rata-rata 24 jam, {shap_df.shape})")
print(f"  -> shap_values_3d.npy  (Mentah, {shap_values_3d.shape})")

# Ringkasan top-5 fitur terpenting
mean_abs_shap = np.abs(shap_values_2d).mean(axis=0)
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Mean |SHAP|': mean_abs_shap
}).sort_values('Mean |SHAP|', ascending=False)

print(f"\nTop-5 Fitur Paling Berpengaruh terhadap PM2.5:")
for i, row in importance_df.head(5).iterrows():
    print(f"  {importance_df.index.get_loc(i)+1}. {row['Feature']:<30s} Mean|SHAP| = {row['Mean |SHAP|']:.6f}")
